# 02e - Shalabi-style images with manually processed good records

Test of the hypothesis that a classifier trained on manually processed good records (ESM25GoodMP) against converted
bad records (ESM25BadCV) separates the classes by the processing (trimming, band-pass filter, taper, common sampling
rate) rather than by the quality. Same images as notebook 02d (`preprocessing.obspy_spectrogram`), except the good
records:

- **good records**: acceleration of ESM25GoodMP (`mp_components`), read record by record from the SeisBench server
  without downloading the file (194 GB), at the sampling rate of the processed data;
- **bad records**: the images of the 02d file, copied (identical settings, nothing to recompute).

Set `preprocessing.obspy_spectrogram.good_source: mp` before running; the file goes to its own folder
(`..._goodmp`), so the 02d file is untouched. The run is **resumable**. For the training (notebook 05) keep
`good_source: mp` and the replication settings of 02d.

## 0. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import shutil
import time

import numpy as np
import pandas as pd

from wqu.config import MP_GOOD_SOURCE, OBSPY_REPRESENTATION, load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.remote import (
    TRACE_NAME_COLUMN,
    component_positions,
    file_component_order,
    iter_remote_trace_batches,
    open_remote_hdf5,
    parse_trace_name,
)
from wqu.data.remote_datasets import remote_dataset_url
from wqu.data.waveforms import stored_sampling_rate_hz
from wqu.preprocessing.obspy_features import (
    DURATION_COLUMN_FORMAT,
    HAS_DATA_COLUMN_FORMAT,
    MP_SAMPLING_RATE_COLUMN,
    SAMPLING_RATE_COLUMN,
    PrecomputedImages,
    check_obspy_features,
    cv_variant,
    excluded_records_path,
    obspy_feature_path,
    store_obspy_features,
    with_mp_sampling_rate,
)
from wqu.preprocessing.pipeline import FEATURE_INDEX_COLUMN, read_feature_records, read_features
from wqu.visualization import apply_plot_style, plot_obspy_images, save_figure

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
WAVEFORM_FILE = "waveforms.hdf5"
BYTES_PER_GB = 1e9
# Size of each range request: large blocks make the sequential reading of the traces faster.
REMOTE_BLOCK_BYTES = 32 * 2**20
TIMING_RECORDS = 256
N_EXAMPLES = 4
COMPONENT_ORDER_COLUMN = "trace_component_order"

config = load_config(CONFIG_PATH)
config = replace(config, data=replace(config.data, source=FULL_SOURCE),
                 waveforms=replace(config.waveforms, source=FULL_SOURCE))
columns = config.columns
preprocessing = config.preprocessing
settings = preprocessing.obspy_spectrogram
if settings is None or settings.good_source != MP_GOOD_SOURCE:
    raise ValueError("Set preprocessing.obspy_spectrogram.good_source: mp in configs/default.yaml.")
label_names = {subset.label: subset.name for subset in config.data.subsets}
good_label = next(subset.label for subset in config.data.subsets if subset.name == "good")
apply_plot_style()

_, metadata = load_esm25_metadata(config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
features_path = obspy_feature_path(preprocessing, FULL_SOURCE)
cv_features_path = obspy_feature_path(cv_variant(preprocessing), FULL_SOURCE)
check_obspy_features(cv_features_path, cv_variant(preprocessing))
mp_url = remote_dataset_url(config.waveforms.remote_url_template, settings.mp_repository_name, WAVEFORM_FILE)
good_records = with_mp_sampling_rate(metadata[metadata[columns.label] == good_label])
bad_records = metadata[metadata[columns.label] != good_label]
figures_dir = features_path.parent / "figures"
print(f"Good records (processed, remote): {len(good_records):,}; bad records (copied): {len(bad_records):,}")
print(f"Images of the bad records from: {cv_features_path}")
print(f"Output: {features_path}")
print(f"Free space: {shutil.disk_usage(Path.home()).free / BYTES_PER_GB:.1f} GB")

## 1. Structure of the remote file

Component order and layout of the traces: the three acceleration components must be found by name, and the stored
traces must have the time axis of the converted ones (same number of samples, zero padding around the processed
part).

In [ ]:
example_names = good_records[TRACE_NAME_COLUMN].iloc[:3].tolist()
with open_remote_hdf5(mp_url, REMOTE_BLOCK_BYTES) as remote:
    stored_order = file_component_order(remote)
    positions = component_positions(stored_order, settings.mp_components)
    block_name, index = parse_trace_name(example_names[0])
    block = remote["data"][block_name]
    print(f"Stored component order: {stored_order}")
    print(f"Positions of {settings.mp_components}: {positions}")
    print(f"Example trace names: {example_names}")
    print(f"Block {block_name}: shape {block.shape}, dtype {block.dtype}, chunks {block.chunks}, "
          f"compression {block.compression}")
if COMPONENT_ORDER_COLUMN in good_records.columns:
    print(f"{COMPONENT_ORDER_COLUMN} in the metadata: {good_records[COMPONENT_ORDER_COLUMN].dropna().unique()[:5]}")

## 2. Reading speed

The first records, in storage order, read with the batch size of `waveforms.batch_size`; the time per record gives the
expected duration of the whole run (reading only, the images add about the time of notebook 02d).

In [ ]:
timing_records = good_records.iloc[:TIMING_RECORDS]
start = time.perf_counter()
read_shapes = []
for batch, traces in iter_remote_trace_batches(timing_records, mp_url, settings.mp_components,
                                               config.waveforms.batch_size, REMOTE_BLOCK_BYTES):
    read_shapes.append(traces.shape)
seconds_per_record = (time.perf_counter() - start) / len(timing_records)
print(f"Batches read: {read_shapes}")
print(f"{seconds_per_record:.3f} s per record; reading all {len(good_records):,} good records: "
      f"about {seconds_per_record * len(good_records) / 3600:.1f} h")
recorded = (np.abs(traces) > config.waveforms.zero_tolerance).any(axis=1).sum(axis=1) / sampling_rate_hz
print(f"Recorded duration of the last batch (s): median {np.median(recorded):.1f}, "
      f"metadata trace_duration_s_orig_mp median {batch['trace_duration_s_orig_mp'].median():.1f}")

## 3. Images (resumable)

The images of the bad records are copied first (a few minutes), then the good records are read from the server and
transformed. The progress bar starts from the records already done.

In [ ]:
records = pd.concat([good_records, bad_records]).loc[metadata.index]
if features_path.exists():
    stored_records = read_feature_records(features_path, columns.text)
    print(f"Feature file already complete: {len(stored_records):,} records with data")
else:
    stored_records = store_obspy_features(
        records,
        lambda selected: iter_remote_trace_batches(selected, mp_url, settings.mp_components,
                                                   config.waveforms.batch_size, REMOTE_BLOCK_BYTES),
        sampling_rate_hz, preprocessing, config.waveforms.zero_tolerance, features_path,
        precomputed=PrecomputedImages(cv_features_path, bad_records.index),
        rate_column=MP_SAMPLING_RATE_COLUMN,
    )
    print(f"Stored {len(stored_records):,} records with data in {features_path}")
excluded_records = pd.read_csv(excluded_records_path(features_path), index_col=0)
print(f"Records without data (left out of records.csv): {len(excluded_records):,}")

## 4. Checks

Per class: records with and without data, sampling rate of the images (processed good records: the rate of the
processed data) and duration of the segments (processed good records: the processed part only).

In [ ]:
counts = pd.DataFrame({
    "with data": stored_records[columns.label].value_counts(),
    "without data": excluded_records[columns.label].value_counts(),
}).fillna(0).astype(int).rename(index=label_names)
display(counts)
rates = (stored_records.groupby(columns.label)[SAMPLING_RATE_COLUMN].value_counts(normalize=True)
         .unstack(level=0).rename(columns=label_names).fillna(0.0))
display(rates.round(4))
duration_columns = [DURATION_COLUMN_FORMAT.format(component) for component in CV_COMPONENTS]
numeric_summary_by_label(stored_records, duration_columns, columns.label).rename(
    index=label_names, level=columns.label).round(1)

## 5. The same good records, converted and processed

Rows in pairs: the image of a good record from the converted data (02d) and from the processed data (this notebook).
Saved in the `figures` folder next to the feature file.

In [ ]:
cv_records = read_feature_records(cv_features_path, columns.text)
good_in_both = stored_records[(stored_records[columns.label] == good_label) & stored_records.index.isin(cv_records.index)]
examples = good_in_both.sample(n=N_EXAMPLES, random_state=config.split.random_seed)
cv_images, _ = read_features(cv_features_path, OBSPY_REPRESENTATION,
                             cv_records.loc[examples.index, FEATURE_INDEX_COLUMN].to_numpy())
mp_images, _ = read_features(features_path, OBSPY_REPRESENTATION, examples[FEATURE_INDEX_COLUMN].to_numpy())
paired = np.stack([image for pair in zip(cv_images, mp_images) for image in pair])
titles = [f"{name} {index}" for index in examples.index for name in ("CV", "MP")]
figure = plot_obspy_images(paired, titles, CV_COMPONENTS, f"Good records, converted and processed: {settings.variant_name}")
save_figure(figure, figures_dir / "cv_and_mp_examples.png")
save_figure(figure, figures_dir / "cv_and_mp_examples.pdf")